# `virtualize()`: combine strategies, single files, and DataTrees

This notebook demonstrates three ways to build virtual datasets with `earthaccess.virtualize()`:

1. **Non-concatenatable granules** — aligning granules by their coordinates (`combine="by_coords"`) or building a synthetic index from the filename with `preprocess`.
2. **Virtualizing a single file** — opening one granule directly, and materializing an index so it can be sliced.
3. **Returning a DataTree** — using VirtualiZarr's `open_virtual_datatree` for multi-group HDF5 granules via `tree=True`.

> These examples exercise the API described in `docs/implementation-plan-virtualize-combine-tree.md`.

## Setup

Log in so earthaccess can build an authenticated registry for indirect (HTTPS) access.

In [ ]:
import earthaccess

earthaccess.login()

## Case 1 — non-concatenatable granules


By default `virtualize()` uses `combine="nested"` and needs a `concat_dim` to stack granules. Some collections' granules do not share a dimension that can be concatenated — instead they must be aligned on their coordinate values, or given a brand-new index.

### 1a. Align by coordinates (`combine="by_coords"`)

`combine="by_coords"` lines the granules up on their shared coordinates (here the per-file `time` stamp) with an outer join.

In [ ]:
sst_granules = earthaccess.search_data(
    concept_id="C1996881146-POCLOUD",
    temporal=("2024-01-01", "2024-01-10"),
)

vds = earthaccess.virtualize(
    sst_granules,
    combine="by_coords",  # no concat_dim needed
    join="outer",
)
vds

### 1b. Build a synthetic index with `preprocess`

If the granules have no dimension to stack, we can create one in `preprocess`. `preprocess` receives each single-granule dataset (so it can read global attributes, but not the file path). Here we promote a timestamp stored as a global attribute to a 1-D `time` index, then stack the granules along it with `combine="nested"`. We also mark `time` as a `loadable_variables` entry so it is materialized as a real array and can be used for label-based slicing.

In [ ]:
import pandas as pd


def add_time_index(ds):
    # The timestamp lives in a global attribute (name varies by
    # collection: time_coverage_start, RangeBeginningDateTime, ...).
    date = pd.Timestamp(ds.attrs["time_coverage_start"])
    return ds.expand_dims("time").assign_coords(time=("time", [date]))

In [ ]:
vds = earthaccess.virtualize(
    sst_granules,
    combine="nested",
    concat_dim="time",
    preprocess=add_time_index,
    loadable_variables=["time"],
)
vds

Because `time` was loaded eagerly, it can be used to select by label without a kerchunk round-trip:

In [ ]:
vds.sel(time="2024-01-03")

## Case 2 — virtualize a single file


A single granule is opened directly with `open_virtual_dataset` (no combine machinery). `loadable_variables` materializes the coordinates we care about while the data variables stay virtual.

In [ ]:
single = earthaccess.virtualize(
    [sst_granules[0]],
    loadable_variables=["time"],
)
single

## Case 3 — return a DataTree (`tree=True`)


Some granules hold multiple HDF5/NetCDF4 groups (TEMPO NO2 has `product` and `geolocation` groups). `tree=True` returns a DataTree with one node per group instead of requiring a manual `xr.merge`.

In [ ]:
tempo = earthaccess.search_data(
    short_name="TEMPO_NO2_L3",
    version="V03",
    temporal=("2024-01-11 12:00", "2024-01-18 12:00"),
    count=1,
)

tree = earthaccess.virtualize([tempo[0]], tree=True)
tree

Navigate the groups just like any xarray DataTree:

In [ ]:
tree["product"]  # the product group
tree["geolocation"]  # the geolocation group

## Summary

| Scenario | How |
| --- | --- |
| Granules can't be concatenated | `combine="by_coords"` + `join` |
| Need an index not in the file | `preprocess` + `combine="nested"` + `loadable_variables` |
| Single file | `virtualize([granule], loadable_variables=[...])` |
| Multi-group HDF5 | `virtualize([granule], tree=True)` |